# Phase 2A — CNN denoising sanity run (Colab T4)

A short pipeline check: **20 training steps maximum**, L1 loss, five small convolutions, and full-page validation PSNR/SSIM. This is not full training or evidence of convergence. The 24 test pairs are never loaded for training, evaluation, or previews.

## Run it yourself
1. Upload this notebook to Google Colab; select **Runtime → Change runtime type → T4 GPU**.
2. Copy these existing project files to Google Drive, preserving the paths below. Use the already downloaded images; no Kaggle login, downloads, or package installation is needed here.
3. Mount Drive in the next cell and set `PROJECT_DIR` to your copied `mini_project` folder. Run cells from top to bottom in a fresh runtime. A missing dependency or GPU stops the notebook rather than installing anything.
4. Keep the defaults for this sanity run. To repeat, restart the runtime and use a new `OUTPUT_DIR` or archive the old output folder yourself.

```text
mini_project/
  src/dataset.py
  src/evaluate.py
  src/models/shallow_cnn.py
  data/raw/train/                 # 144 existing noisy PNGs
  data/raw/train_cleaned/         # 144 existing clean PNGs
  outputs/phase1/split_manifest.json
```

The checkpoint, metrics JSON, step log, loss plot and Noisy / Denoised / Clean comparisons are saved to `outputs/phase2a_sanity`. Copying the project to Drive keeps these after a Colab session ends. Do not upload the old filename-split audit as a replacement for `split_manifest.json`.

In [ ]:
# Colab only: authorize your own Drive mount. No data is downloaded.
from google.colab import drive
drive.mount('/content/drive')

## Setup and fixed sanity budget
Batch size 1 reuses the existing loader for variable-size pages. Only after selecting training IDs do we take matching 256×256 crops. Validation uses all 20 native-size pages. AMP is enabled on CUDA; metrics use float32. The notebook reports peak allocated and reserved PyTorch memory and stops if either exceeds 6 GiB (this excludes other processes).

In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'  # Set before CUDA work.
from pathlib import Path
import sys, json, hashlib, random, time, itertools
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from PIL import Image
import struct

PROJECT_DIR = Path('/content/drive/MyDrive/AIE464/mini_project')  # EDIT THIS.
OUTPUT_DIR = PROJECT_DIR / 'outputs/phase2a_sanity'
SEED, MAX_STEPS, PATCH_SIZE, LEARNING_RATE, WIDTH = 42, 20, 256, 1e-3, 16
REQUIRE_CUDA = True  # False only for a local CPU smoke check.
MEMORY_LIMIT_BYTES = 6 * 1024**3
EXPECTED_MANIFEST_SHA256 = '5b569d65a4d21aacff84e25986fce522847ce61a4575cd720cd60ca4a4180539'
assert 1 <= MAX_STEPS <= 20, 'Phase 2A is capped at 20 steps; no full training.'
assert PATCH_SIZE == 256 and WIDTH == 16, 'Keep the minimal sanity configuration.'
if REQUIRE_CUDA and not torch.cuda.is_available():
    raise RuntimeError('Select a Colab T4 GPU runtime, then restart and run all cells.')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = device.type == 'cuda'
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if AMP:
    torch.cuda.manual_seed_all(SEED)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False
torch.use_deterministic_algorithms(True)
if (OUTPUT_DIR / 'checkpoint.pt').exists():
    raise FileExistsError('Checkpoint already exists. Use a new OUTPUT_DIR for a fresh run.')
required = ['src/dataset.py', 'src/evaluate.py', 'src/models/shallow_cnn.py',
            'outputs/phase1/split_manifest.json']
for relative in required:
    if not (PROJECT_DIR / relative).is_file():
        raise FileNotFoundError(PROJECT_DIR / relative)
sys.path.insert(0, str(PROJECT_DIR / 'src'))
from dataset import PairedDocumentDataset, make_loader
from models.shallow_cnn import ShallowCNN
from evaluate import image_metrics

def synchronize():
    if AMP:
        torch.cuda.synchronize()

def gpu_memory():
    if not AMP:
        return {'peak_allocated_mib': 0.0, 'peak_reserved_mib': 0.0}
    allocated = torch.cuda.max_memory_allocated()
    reserved = torch.cuda.max_memory_reserved()
    if max(allocated, reserved) > MEMORY_LIMIT_BYTES:
        raise RuntimeError('PyTorch peak GPU memory exceeded 6 GiB; stop this sanity run.')
    return {'peak_allocated_mib': allocated / 1024**2,
            'peak_reserved_mib': reserved / 1024**2}

print('PyTorch:', torch.__version__, '| device:', device,
      '| GPU:', torch.cuda.get_device_name() if AMP else 'CPU', '| AMP:', AMP)

## Reuse the frozen group-aware manifest
Verify its checksum and group separation, without regenerating splits. Only the train and validation images are decoded below. Test filenames are read as metadata solely to assert disjoint assignments. All four variants of an identical clean document remain together.

In [ ]:
manifest_path = PROJECT_DIR / 'outputs/phase1/split_manifest.json'
manifest_bytes = manifest_path.read_bytes()
manifest_sha256 = hashlib.sha256(manifest_bytes).hexdigest()
assert manifest_sha256 == EXPECTED_MANIFEST_SHA256, 'Unexpected manifest; use the verified Phase 1 file.'
manifest = json.loads(manifest_bytes)
splits, group_splits, groups = manifest['splits'], manifest['group_splits'], manifest['groups']
for name, expected_groups, expected_pairs in [('train', 25, 100), ('val', 5, 20), ('test', 6, 24)]:
    assert len(group_splits[name]) == len(set(group_splits[name])) == expected_groups
    assert len(splits[name]) == len(set(splits[name])) == expected_pairs
    assert {p for g in group_splits[name] for p in groups[g]} == set(splits[name])
    assert all(len(groups[g]) == 4 for g in group_splits[name])
for left, right in itertools.combinations(splits, 2):
    assert not set(splits[left]) & set(splits[right])
    assert not set(group_splits[left]) & set(group_splits[right])
noisy_dir, clean_dir = PROJECT_DIR / 'data/raw/train', PROJECT_DIR / 'data/raw/train_cleaned'
train_dataset = PairedDocumentDataset(noisy_dir, clean_dir, splits['train'])
val_dataset = PairedDocumentDataset(noisy_dir, clean_dir, splits['val'])
# Verify actual clean content for train/val against the saved fingerprints.
for name in ('train', 'val'):
    for group_id in group_splits[name]:
        for page_id in groups[group_id]:
            with Image.open(clean_dir / page_id) as image:
                pixels = np.array(image.convert('L'), dtype=np.uint8)
            h, w = pixels.shape
            actual = hashlib.sha256(struct.pack('>II', w, h) + pixels.tobytes()).hexdigest()
            assert actual == group_id, f'Clean data changed: {page_id}'
train_loader = make_loader(train_dataset, shuffle=True, seed=SEED)
val_loader = make_loader(val_dataset)
preview_ids = {groups[g][0] for g in group_splits['val'][:3]}  # Three distinct val groups.
print('Train:', len(train_dataset), '| Validation:', len(val_dataset),
      '| Test: reserved, not loaded | group overlap: 0')

## Brief training — training split only
The model preserves spatial size with stride 1 and padding 1. Sigmoid bounds its output to [0,1]. Adam minimizes L1 on exactly one shared crop per input/target pair. There are no epochs, scheduling, early stopping, or hyperparameter searches. The final checkpoint is used regardless of validation scores.

In [ ]:
synchronize()
run_started = time.perf_counter()
model = ShallowCNN(width=WIDTH).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
criterion = nn.L1Loss()
scaler = torch.amp.GradScaler('cuda', enabled=AMP)
crop_generator = torch.Generator().manual_seed(SEED)
history = []
model.train()
synchronize()
train_started = time.perf_counter()
for step, batch in enumerate(itertools.islice(train_loader, MAX_STEPS), start=1):
    noisy, clean = batch['noisy'], batch['clean']
    h, w = noisy.shape[-2:]
    assert min(h, w) >= PATCH_SIZE
    top = int(torch.randint(h - PATCH_SIZE + 1, (1,), generator=crop_generator))
    left = int(torch.randint(w - PATCH_SIZE + 1, (1,), generator=crop_generator))
    noisy = noisy[..., top:top + PATCH_SIZE, left:left + PATCH_SIZE].contiguous().to(device)
    clean = clean[..., top:top + PATCH_SIZE, left:left + PATCH_SIZE].contiguous().to(device)
    optimizer.zero_grad(set_to_none=True)
    with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
        denoised = model(noisy)
        assert denoised.shape == clean.shape
        loss = criterion(denoised, clean)
    if not torch.isfinite(loss):
        raise RuntimeError('Non-finite training loss')
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0, error_if_nonfinite=True)
    scaler.step(optimizer)
    scaler.update()
    synchronize()
    record = {'step': step, 'id': batch['id'][0], 'crop_top': top, 'crop_left': left,
              'l1_loss': loss.item(), 'elapsed_seconds': time.perf_counter() - train_started,
              **gpu_memory()}
    history.append(record)
    print(f"step {step:02d}/{MAX_STEPS} | L1={record['l1_loss']:.6f} | "
          f"elapsed={record['elapsed_seconds']:.2f}s | peak={record['peak_allocated_mib']:.1f} MiB")
assert len(history) == MAX_STEPS
synchronize()
train_seconds = time.perf_counter() - train_started
training_memory = gpu_memory()
print('Parameters:', sum(p.numel() for p in model.parameters()))

## Validation — full pages, no test data
Report mean **per-page** scores over the same 20 validation pairs for noisy input and CNN output. PSNR uses data range 1; SSIM uses an 11×11 Gaussian window (sigma 1.5), population covariance and valid windows, with no downsampling. The model switches to evaluation mode and gradients are disabled. Float32 metrics are calculated outside AMP. A short sanity run may produce worse results than the noisy input.

In [ ]:
model.eval()
validation_rows, comparisons = [], []
synchronize()
val_started = time.perf_counter()
with torch.inference_mode():
    for batch in val_loader:
        noisy, clean = batch['noisy'].to(device), batch['clean'].to(device)
        with torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
            denoised = model(noisy)
        denoised = denoised.float()
        assert denoised.shape == clean.shape
        noisy_scores = image_metrics(noisy, clean)
        denoised_scores = image_metrics(denoised, clean)
        validation_rows.append({'id': batch['id'][0],
            'noisy_psnr': noisy_scores['psnr'].item(), 'noisy_ssim': noisy_scores['ssim'].item(),
            'denoised_psnr': denoised_scores['psnr'].item(), 'denoised_ssim': denoised_scores['ssim'].item(),
            'denoised_l1': criterion(denoised, clean).item()})
        gpu_memory()
        if batch['id'][0] in preview_ids:
            comparisons.append({'id': batch['id'][0], 'noisy': noisy.cpu(),
                                'denoised': denoised.cpu(), 'clean': clean.cpu()})
synchronize()
val_seconds = time.perf_counter() - val_started
assert len(validation_rows) == 20 and len(comparisons) == 3
mean_metrics = {key: float(np.mean([row[key] for row in validation_rows]))
                for key in validation_rows[0] if key != 'id'}
for key, value in mean_metrics.items():
    print(f'{key}: {value:.6f}')
print(f'Train: {train_seconds:.2f}s | validation: {val_seconds:.2f}s | memory: {gpu_memory()}')

## Save a reproducible sanity checkpoint and logs
Store final weights, optimizer/scaler states, seeds/RNG states, actual trained page/crop order, module hashes, data hashes for train/val, the exact split manifest and runtime versions. Restarting and running all cells reproduces the experiment in the same software/hardware environment; exact results across different PyTorch/CUDA versions or devices are not promised. This is a final sanity checkpoint, not a claim of convergence or an automatic mid-epoch resume mechanism.

In [ ]:
synchronize()
summary = {'phase': '2A sanity only', 'steps': len(history), 'train_seconds': train_seconds,
           'validation_seconds': val_seconds, 'run_seconds': time.perf_counter() - run_started,
           'training_gpu_memory': training_memory, 'overall_gpu_memory': gpu_memory(),
           'mean_validation_metrics': mean_metrics, 'manifest_sha256': manifest_sha256,
           'test_evaluated': False}
config = {'seed': SEED, 'max_steps': MAX_STEPS, 'batch_size': 1, 'patch_size': PATCH_SIZE,
          'learning_rate': LEARNING_RATE, 'width': WIDTH, 'loss': 'L1', 'optimizer': 'Adam',
          'amp': AMP, 'gradient_clip_norm': 1.0, 'memory_limit_bytes': MEMORY_LIMIT_BYTES,
          'ssim': 'Gaussian11 sigma1.5 population covariance valid windows data_range1'}
versions = {'torch': str(torch.__version__), 'numpy': np.__version__,
            'cuda': torch.version.cuda, 'cudnn': torch.backends.cudnn.version(),
            'device': str(device), 'gpu': torch.cuda.get_device_name() if AMP else None}
source_hashes = {relative: hashlib.sha256((PROJECT_DIR / relative).read_bytes()).hexdigest()
                 for relative in required if relative.endswith('.py')}
data_hashes = {f'{folder}/{page_id}': hashlib.sha256((PROJECT_DIR / f'data/raw/{folder}' / page_id).read_bytes()).hexdigest()
               for folder in ('train', 'train_cleaned') for name in ('train', 'val') for page_id in splits[name]}
numpy_state = np.random.get_state()
checkpoint = {'model_state_dict': {k: v.detach().cpu() for k, v in model.state_dict().items()},
              'optimizer_state_dict': optimizer.state_dict(), 'scaler_state_dict': scaler.state_dict(),
              'config': config, 'versions': versions, 'source_sha256': source_hashes,
              'data_sha256': data_hashes, 'split_manifest': manifest, 'summary': summary,
              'history': history, 'validation_rows': validation_rows,
              'rng': {'torch_cpu': torch.get_rng_state(),
                      'torch_cuda': torch.cuda.get_rng_state_all() if AMP else [],
                      'crop_generator': crop_generator.get_state(),
                      'loader_generator': train_loader.generator.get_state(),
                      'python': random.getstate(),
                      'numpy': [numpy_state[0], numpy_state[1].tolist(), *numpy_state[2:]]}}
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(checkpoint, OUTPUT_DIR / 'checkpoint.pt')
(OUTPUT_DIR / 'step_log.json').write_text(json.dumps(history, indent=2) + '\n')
(OUTPUT_DIR / 'metrics.json').write_text(json.dumps({'summary': summary, 'config': config,
    'versions': versions, 'validation_rows': validation_rows}, indent=2) + '\n')
print('Saved checkpoint and logs to', OUTPUT_DIR)

## Loss and Noisy / Denoised / Clean comparisons
Three examples come from three distinct validation clean-document groups. These pictures and scores diagnose the pipeline; they do not justify model selection on the reserved test set.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3))
ax.plot([r['step'] for r in history], [r['l1_loss'] for r in history], marker='o')
ax.set(xlabel='Training step', ylabel='L1 loss', title='Phase 2A sanity loss')
fig.tight_layout(); fig.savefig(OUTPUT_DIR / 'loss.png', dpi=140)
plt.show(); plt.close(fig)
fig, axes = plt.subplots(len(comparisons), 3, figsize=(12, 9), constrained_layout=True)
for row, sample in enumerate(comparisons):
    for col, key in enumerate(('noisy', 'denoised', 'clean')):
        axes[row, col].imshow(sample[key][0, 0].numpy(), cmap='gray', vmin=0, vmax=1)
        axes[row, col].set_title(f"{sample['id']} | {key.title()}")
        axes[row, col].axis('off')
fig.savefig(OUTPUT_DIR / 'comparisons.png', dpi=140)
plt.show(); plt.close(fig)

# Reload final weights and verify the same validation prediction.
saved = torch.load(OUTPUT_DIR / 'checkpoint.pt', map_location='cpu', weights_only=True)
restored = ShallowCNN(width=saved['config']['width']).to(device)
restored.load_state_dict(saved['model_state_dict'])
restored.eval()
with torch.inference_mode(), torch.amp.autocast(device_type=device.type, dtype=torch.float16, enabled=AMP):
    restored_prediction = restored(comparisons[0]['noisy'].to(device)).float().cpu()
torch.testing.assert_close(restored_prediction, comparisons[0]['denoised'], rtol=0, atol=0)
print('Checkpoint reload: identical validation prediction. Sanity run finished.')

## Interpretation and references
Successful checks mean that grouped loading, aligned crops, gradients, validation metrics and checkpoint saving work. **20 steps are not sufficient to claim model quality or convergence.** Keep the test split untouched for later shared benchmarks. PyTorch reports allocated/reserved tensor memory, not other GPU processes or all driver overhead; inspect Colab GPU usage separately if needed.

The SSIM helper implements the stated Gaussian-window formula without extra dependencies. References: [PyTorch AMP](https://docs.pytorch.org/docs/2.8/amp.html), [PyTorch reproducibility](https://docs.pytorch.org/docs/2.8/notes/randomness.html), [original SSIM paper](https://ece.uwaterloo.ca/~z70wang/publications/ssim.html).

Implementation verification is recorded in `docs/PHASE2A_SANITY.md`. The delivered notebook has empty outputs; Colab T4 results are generated when you run it.